In [2]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import category_encoders as ce
import shap


# ─────────────────────────────────────────────────────────────
# STEP 1 — Load and prepare data
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Loading and preparing data")
print("=" * 60)

df = pd.read_excel('microrna_families_conserved.xlsx') 

df['parasite'] = df['parasite'].str.replace(r'\s+', '', regex=True)

# ── Build lookup dicts 
df_raw = df.copy()

mirna_lookup = (
    df_raw[['microrna', 'microrna_group_simplified', 'family_name', 'mirbase_accession']]
    .drop_duplicates('microrna')
    .set_index('microrna')
    .to_dict('index')
)

accession_lookup = (
    df_raw[['mirbase_accession', 'microrna', 'microrna_group_simplified', 'family_name']] 
    .dropna(subset=['mirbase_accession'])
    .drop_duplicates('mirbase_accession')
    .set_index('mirbase_accession')
    .to_dict('index')
)

print(f"miRNA lookup     : {len(mirna_lookup)} unique miRNA names")
print(f"Accession lookup : {len(accession_lookup)} unique accession numbers")

df = df.drop(columns=['microrna', 'infection', 'microrna_group_simplified', 'mirbase_accession'])

df['family_name'] = df['family_name'].replace('not_found', np.nan)   
df['is_conserved'] = df['family_name'].notna().astype(int)           
df['parasite_celltype'] = df['parasite'].astype(str) + '_' + df['cell type'].astype(str)

print(f"Total rows            : {len(df)}")
print(f"Family known          : {df['is_conserved'].sum()}")
print(f"Family unknown        : {(df['is_conserved']==0).sum()}")
print(f"Target balance        :\n{df['is_upregulated'].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────
# STEP 2 — Features and target
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Features and target")
print("=" * 60)

TARGET   = 'is_upregulated'
CAT_COLS = ['parasite', 'organism', 'cell type', 'family_name', 'parasite_celltype']  
NUM_COLS = ['time', 'is_conserved']

X = df[CAT_COLS + NUM_COLS]
y = df[TARGET]

print(f"Categorical   : {CAT_COLS}")
print(f"Numeric       : {NUM_COLS}")
print(f"Total features: {len(CAT_COLS) + len(NUM_COLS)}")


# ─────────────────────────────────────────────────────────────
# STEP 3 — Optuna hyperparameter search
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Optuna hyperparameter search (100 trials)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-4, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 8, 64),
        'min_child_samples': trial.suggest_int('min_child_samples', 2, 30),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-6, 10.0, log=True),
        'random_state':      42,
        'verbose':           -1,
        'class_weight':      'balanced',
    }

    pipe = Pipeline([
        ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
        ('model',   LGBMClassifier(**params))
    ])

    scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
    return scores.mean()


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

BEST_PARAMS = study.best_params
BEST_PARAMS.update({'random_state': 42, 'verbose': -1, 'class_weight': 'balanced'})

print(f"\nBest AUC (Optuna)  : {study.best_value:.4f}")
print(f"Best params found  :")
for k, v in BEST_PARAMS.items():
    print(f"  {k:<25}: {v}")

trials_df = study.trials_dataframe().sort_values('value', ascending=False)
print(f"\nTop 5 trials:")
print(trials_df[['number', 'value']].head(5).to_string(index=False))


# ─────────────────────────────────────────────────────────────
# STEP 4 — Cross-validate with best params
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Cross-Validation with new best params (5-fold)")
print("=" * 60)

pipe_best = Pipeline([
    ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
    ('model',   LGBMClassifier(**BEST_PARAMS))
])

auc = cross_val_score(pipe_best, X, y, cv=cv, scoring='roc_auc')
acc = cross_val_score(pipe_best, X, y, cv=cv, scoring='accuracy')
f1  = cross_val_score(pipe_best, X, y, cv=cv, scoring='f1')

print(f"ROC-AUC : {auc.mean():.3f} ± {auc.std():.3f}")
print(f"Accuracy: {acc.mean():.3f} ± {acc.std():.3f}")
print(f"F1      : {f1.mean():.3f} ± {f1.std():.3f}")
print(f"AUC per fold: {[round(x, 3) for x in auc.tolist()]}")


# ─────────────────────────────────────────────────────────────
# STEP 5 — Train final model on all data
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Training final model on all data")
print("=" * 60)

pipe_best.fit(X, y)
print("Done.")


# ─────────────────────────────────────────────────────────────
# STEP 6 — Permutation Feature Importance
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Permutation Feature Importance")
print("=" * 60)

X_enc = pipe_best.named_steps['encoder'].transform(X)
perm  = permutation_importance(
    pipe_best.named_steps['model'],
    X_enc, y, n_repeats=30, random_state=42, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature':    X.columns.tolist(),
    'importance': perm.importances_mean,
    'std':        perm.importances_std
}).sort_values('importance', ascending=False)

print(f"\n{'Feature':<25} {'Importance':>10} {'Std':>8}")
print("-" * 47)
for _, row in perm_df.iterrows():
    bar  = '█' * max(0, int(row['importance'] * 60))
    sign = '+' if row['importance'] >= 0 else ''
    print(f"  {row['feature']:<23} {sign}{row['importance']:.4f} ± {row['std']:.4f}  {bar}")


# ─────────────────────────────────────────────────────────────
# STEP 7 — Save model bundle
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Saving model bundle")
print("=" * 60)

model_bundle = {
    'model':     pipe_best,
    'encoder':   pipe_best.named_steps['encoder'],
    'lgbm':      pipe_best.named_steps['model'],
    'metrics': {
        'auc_mean':        round(auc.mean(), 3),
        'auc_std':         round(auc.std(),  3),
        'acc_mean':        round(acc.mean(), 3),
        'acc_std':         round(acc.std(),  3),
        'f1_mean':         round(f1.mean(),  3),
        'f1_std':          round(f1.std(),   3),
        'auc_folds':       [round(x, 3) for x in auc.tolist()],
        'best_params':     BEST_PARAMS,
        'optuna_best_auc': round(study.best_value, 4),
        'n_trials':        100,
        'n_train':         len(X),
        'feature_importance': perm_df.to_dict('records'),
    },
    'options': {
        'parasite':      sorted(df['parasite'].dropna().unique().tolist()),
        'organism':      sorted(df['organism'].dropna().unique().tolist()),
        'cell_type':     sorted(df['cell type'].dropna().unique().tolist()),
        'time':          sorted(df['time'].dropna().unique().tolist()),
        'seed_families': sorted(df['family_name'].dropna().unique().tolist()),  
    },
    'cat_cols':      CAT_COLS,
    'feature_names': list(X.columns),
    'mirna_lookup':      mirna_lookup,
    'accession_lookup':  accession_lookup,
}

with open('Model206_conserved_model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
print("Saved: Model206_accession_model.pkl")

STEP 1 — Loading and preparing data
miRNA lookup     : 115 unique miRNA names
Accession lookup : 59 unique accession numbers
Total rows            : 206
Family known          : 120
Family unknown        : 86
Target balance        :
is_upregulated
1    103
0    103

STEP 2 — Features and target
Categorical   : ['parasite', 'organism', 'cell type', 'family_name', 'parasite_celltype']
Numeric       : ['time', 'is_conserved']
Total features: 7

STEP 3 — Optuna hyperparameter search (100 trials)


  0%|          | 0/100 [00:00<?, ?it/s]


Best AUC (Optuna)  : 0.8129
Best params found  :
  n_estimators             : 324
  max_depth                : 5
  learning_rate            : 0.01233422752240475
  num_leaves               : 42
  min_child_samples        : 6
  subsample                : 0.9340403835363337
  colsample_bytree         : 0.5463598755863698
  reg_alpha                : 0.0002462088010773862
  reg_lambda               : 0.0008915890327903716
  random_state             : 42
  verbose                  : -1
  class_weight             : balanced

Top 5 trials:
 number    value
     74 0.812914
     91 0.812653
     61 0.811508
     98 0.811009
     75 0.811009

STEP 4 — Cross-Validation with new best params (5-fold)
ROC-AUC : 0.813 ± 0.063
Accuracy: 0.728 ± 0.055
F1      : 0.729 ± 0.041
AUC per fold: [0.875, 0.888, 0.792, 0.714, 0.795]

STEP 5 — Training final model on all data
Done.

STEP 6 — Permutation Feature Importance

Feature                   Importance      Std
-----------------------------------------